## Column-by-column plan

| Column | What it represents | Cleaning rule |
|---|---|---|
| `inpatient_number` | Identifier | Read as text. Check that it exists and does not repeat. |
| `eye_opening` | GCS eye component | Convert to integer; require 1–4. |
| `verbal_response` | GCS verbal component | Convert to integer; require 1–5. |
| `movement` | GCS motor component | Convert to integer; require 1–6. |
| `consciousness` | Recorded category | Normalize whitespace and capitalization; preserve the recorded category. |
| `gcs` | GCS total | Convert to integer; require 3–15 and compare with the sum of the three components. |

For any missing, unfamiliar, or contradictory score, the code stops and asks for source review. It does not create a clinical value by imputation. A separate review flag identifies category patterns that require context outside this file.


## Step 1 — Load the workbook without altering it

**What the code does:** Finds the Excel file, reads the `responsivenes` sheet, and creates `clean`, an independent copy of `raw`. It reads `inpatient_number` as text because IDs are labels, not numbers to average.

In [1]:
from pathlib import Path
import pandas as pd

# Try the two common locations for the input workbook.
candidates = [Path('responsivenes.xlsx'), Path('upload/responsivenes.xlsx')]
source = next((path for path in candidates if path.exists()), None)
if source is None:
    raise FileNotFoundError('Place responsivenes.xlsx beside this notebook or in upload/.')

raw = pd.read_excel(source, sheet_name='responsivenes', dtype={'inpatient_number': 'string'})
clean = raw.copy(deep=True)  # Preserve raw for comparison.
print(f'Source: {source.name}; rows: {len(raw):,}; columns: {len(raw.columns)}')
print('Column names:', raw.columns.tolist())
print('Starting data types:')
print(raw.dtypes.to_string())

Source: responsivenes.xlsx; rows: 2,008; columns: 6
Column names: ['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs']
Starting data types:
inpatient_number    string[python]
eye_opening                  int64
verbal_response              int64
movement                     int64
consciousness               object
gcs                          int64


## Step 2 — Standardize names and check missing data

**What the code does:** Trims and standardizes column headers; trims the identifier and category text; turns empty text into missing values; counts missing cells in every column.

In [2]:
# Turn any irregular header spacing into predictable snake_case names.
clean.columns = (clean.columns.astype(str).str.strip().str.lower()
                 .str.replace(r'\s+', '_', regex=True))
expected = ['inpatient_number', 'eye_opening', 'verbal_response',
            'movement', 'consciousness', 'gcs']
if clean.columns.tolist() != expected:
    raise ValueError(f'Unexpected columns: {clean.columns.tolist()}')

# Trim identifier and category text; blanks become missing values.
clean['inpatient_number'] = clean['inpatient_number'].str.strip()
clean['consciousness'] = clean['consciousness'].astype('string').str.strip()
clean[['inpatient_number', 'consciousness']] = (
    clean[['inpatient_number', 'consciousness']].replace('', pd.NA)
)
missing = clean.isna().sum()
print('Missing values per column:')
print(missing.to_string())
header_changes = sum(a != b for a, b in zip(raw.columns, clean.columns))
print('Header names changed:', header_changes)
if missing.any():
    raise ValueError('Missing values need case-by-case review before export.')

Missing values per column:
inpatient_number    0
eye_opening         0
verbal_response     0
movement            0
consciousness       0
gcs                 0
Header names changed: 0


## Step 3 — Find duplicate records and IDs

**What the code does:** Counts and removes rows that are identical across all six input fields. It separately counts repeated `inpatient_number` values without deleting them.

In [3]:
# Only completely identical records are safe to deduplicate.
exact_duplicates = int(clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)
# Review repeated IDs separately rather than silently deleting them.
duplicate_ids = int(clean['inpatient_number'].duplicated().sum())
print(f'Exact duplicate rows removed: {exact_duplicates}')
print(f'Repeated inpatient_number values: {duplicate_ids}')
if duplicate_ids:
    raise ValueError('Repeated identifiers need review before export.')

Exact duplicate rows removed: 0
Repeated inpatient_number values: 0


## Step 4 — Convert and validate the four GCS score columns

**What the code does:** Converts each score to a whole number, then checks its permitted interval: eye 1–4, verbal 1–5, movement 1–6, total 3–15. It rejects text that cannot be parsed, fractions, and values outside the interval.

In [4]:
ranges = {'eye_opening': (1, 4), 'verbal_response': (1, 5),
          'movement': (1, 6), 'gcs': (3, 15)}
for column, (minimum, maximum) in ranges.items():
    # Nonnumeric source entries become NaN so they can be counted.
    converted = pd.to_numeric(clean[column], errors='coerce')
    # Reject missing conversions, fractions, and out-of-range scores.
    invalid = converted.isna() | (converted % 1 != 0) | ~converted.between(minimum, maximum)
    if invalid.any():
        raise ValueError(f'{column}: {int(invalid.sum())} invalid score(s); inspect the source.')
    clean[column] = converted.astype('int64')
    print(f'{column}: observed {clean[column].min()}–{clean[column].max()}; '
          f'invalid values: {int(invalid.sum())}')

eye_opening: observed 1–4; invalid values: 0
verbal_response: observed 1–5; invalid values: 0
movement: observed 1–6; invalid values: 0
gcs: observed 3–15; invalid values: 0


## Step 5 — Normalize the recorded category

**What the code does:** Removes extra spaces and standardizes capitalization against the four category names actually present in the dataset. An unfamiliar category causes a clear error.

In [5]:
category_map = {
    'clear': 'Clear',
    'responsivetosound': 'ResponsiveToSound',
    'responsivetopain': 'ResponsiveToPain',
    'nonresponsive': 'Nonresponsive',
}
# Keep the current labels so the formatting-change count can be audited.
before_labels = clean['consciousness'].copy()
normalized_key = clean['consciousness'].str.replace(r'\s+', '', regex=True).str.casefold()
normalized_labels = normalized_key.map(category_map)
if normalized_labels.isna().any():
    raise ValueError('Unknown consciousness labels need manual review before export.')
clean['consciousness'] = normalized_labels.astype('string')
print('Category labels changed only by formatting:', int((before_labels != clean['consciousness']).sum()))
print('Category totals:')
print(clean['consciousness'].value_counts().to_string())

Category labels changed only by formatting: 0
Category totals:
consciousness
Clear                1974
ResponsiveToSound      19
Nonresponsive          11
ResponsiveToPain        4


## Step 6 — Verify the total GCS in every row

**What the code does:** Calculates `eye_opening + verbal_response + movement` and compares it with the recorded `gcs` for each patient.

In [6]:
# The total must equal the sum of the three component scores.
calculated_gcs = clean[['eye_opening', 'verbal_response', 'movement']].sum(axis=1)
gcs_mismatches = int((calculated_gcs != clean['gcs']).sum())
print(f'Rows where recorded GCS differs from its components: {gcs_mismatches}')
if gcs_mismatches:
    raise ValueError('GCS discrepancies need source review before export.')

Rows where recorded GCS differs from its components: 0


## Step 7 — Flag category differences that need human review

**What the code does:** Groups records by the same three GCS component scores. If a combination occurs with multiple `consciousness` labels, it marks **every** record in that combination with `consciousness_review_flag=True`.

In [7]:
component_cols = ['eye_opening', 'verbal_response', 'movement']
# Compare labels only within rows with the same exact component scores.
number_of_labels = clean.groupby(component_cols)['consciousness'].transform('nunique')
clean['consciousness_review_flag'] = number_of_labels.gt(1)
ambiguous_patterns = clean.loc[clean['consciousness_review_flag'], component_cols].drop_duplicates()
print(f'Score combinations with multiple labels: {len(ambiguous_patterns)}')
print(f'Rows flagged for label review: {int(clean["consciousness_review_flag"].sum())}')
print('Review groups (aggregate counts; no identifiers):')
print(clean.loc[clean['consciousness_review_flag']].groupby(component_cols)
      .agg(records=('gcs', 'size'), different_labels=('consciousness', 'nunique'))
      .reset_index().to_string(index=False))

Score combinations with multiple labels: 2
Rows flagged for label review: 29
Review groups (aggregate counts; no identifiers):
 eye_opening  verbal_response  movement  records  different_labels
           1                1         1       13                 4
           3                3         5       16                 2


## Step 8 — Record the cleaning decisions and save the result

**What the code does:** Prints an audit table of issues found and changes made, exports the seven-column table to `responsivenes_cleaned.csv`, and reads it back to check its row count, identifiers, flag count, and columns.

In [8]:
summary = pd.DataFrame([
    ('Input rows', len(raw), 'Keep source record count for reconciliation'),
    ('Rows exported', len(clean), 'No records removed in this file'),
    ('Exact duplicates removed', exact_duplicates, 'Only exact copies would be removed'),
    ('Missing cells', int(missing.sum()), 'No imputation needed'),
    ('Duplicate patient IDs', duplicate_ids, 'No identifier conflicts found'),
    ('GCS sum mismatches', gcs_mismatches, 'No recorded totals overwritten'),
    ('Header names changed', header_changes, 'Schema checked after normalization'),
    ('Category formatting changes', int((before_labels != clean['consciousness']).sum()),
     'Original label meaning retained'),
    ('Rows flagged for label review', int(clean['consciousness_review_flag'].sum()),
     'Retained with a review flag; no guessed relabeling'),
], columns=['Check', 'Count', 'Decision'])
display(summary)
print('Final data types:')
print(clean.dtypes.to_string())

output = Path('responsivenes_cleaned.csv')
clean.to_csv(output, index=False)

# Read it back to check that the exported rows, IDs, and flag survived.
reloaded = pd.read_csv(output, dtype={'inpatient_number': 'string'})
assert len(reloaded) == len(clean)
assert reloaded['inpatient_number'].equals(clean['inpatient_number'])
assert int(reloaded['consciousness_review_flag'].sum()) == int(clean['consciousness_review_flag'].sum())
assert reloaded.columns.tolist() == clean.columns.tolist()
print(f'Saved and verified: {output.name} ({len(reloaded):,} rows, {len(reloaded.columns)} columns)')

                        Check  Count                                           Decision
                   Input rows   2008        Keep source record count for reconciliation
                Rows exported   2008                    No records removed in this file
     Exact duplicates removed      0                 Only exact copies would be removed
                Missing cells      0                               No imputation needed
        Duplicate patient IDs      0                      No identifier conflicts found
           GCS sum mismatches      0                     No recorded totals overwritten
         Header names changed      0                 Schema checked after normalization
  Category formatting changes      0                    Original label meaning retained
Rows flagged for label review     29 Retained with a review flag; no guessed relabeling
Final data types:
inpatient_number             string[python]
eye_opening                           int64
verbal_respons

### Overall conclusion

I audited the 2,008 responsiveness records for missing data, repeated patient IDs, duplicate rows, invalid GCS component ranges, and incorrect GCS totals. All values passed those checks, so I preserved the original records. Two identical score patterns carried different consciousness labels across 29 records; I kept the recorded labels and marked those rows for review instead of guessing a correction. The notebook exports a reproducible CSV with that review flag.